# Biohub Cell Tracking — Submission Skeleton (DRAFT, not for submit)

Status: **skeleton** (GOLD §5). Wiring only — detection hook is a synthetic demo until real weights land.
Constraints: Kaggle notebook, **no internet**, GPU (T4), **≤12 h** total, ~80–96 s/video.
Trusted logic mirrors repo `scripts/` at PROTOCOL v1.1: voxel `z=1.625/y=x=0.40625`, 7 µm match gate, causal adjacent-frame Hungarian links, fork proposals `propose-um=10` (EXP-0005 r10). Do not retune gates here — change them in the repo, re-validate, then copy.

TODO before any serious submit:
1. Replace `detect()` demo with the real detector + bundled weights (attach via Kaggle datasets, load from `/kaggle/input/`).
2. Flip `MODE` to `"test"` and point `TEST_ROOT` at the hidden-test mount.
3. Dry-run the full notebook end-to-end and confirm per-video time × N fits 12 h with ≥2× headroom (PROTOCOL §7).

In [ ]:
MODE = "demo"  # "demo" (synthetic wiring check) | "test" (hidden test set)
TEST_ROOT = "/kaggle/input/biohub-cell-tracking-during-development/test"
OUT_CSV = "/kaggle/working/submission.csv"
VOXEL = (1.625, 0.40625, 0.40625)
MATCH_UM, PROPOSE_UM = 7.0, 10.0  # frozen per PROTOCOL v1.1 / EXP-0005 r10
import time; T0 = time.time()

In [ ]:
import numpy as np

def hungarian(cost):
    n = len(cost)
    if n == 0: return []
    u, v, p, way = [0.]*(n+1), [0.]*(n+1), [0]*(n+1), [0]*(n+1)
    for i in range(1, n+1):
        p[0] = i; j0 = 0
        minv, used = [float('inf')]*(n+1), [False]*(n+1)
        while True:
            used[j0] = True; i0 = p[j0]; delta, j1 = float('inf'), 0
            for j in range(1, n+1):
                if used[j]: continue
                cur = cost[i0-1][j-1]-u[i0]-v[j]
                if cur < minv[j]: minv[j], way[j] = cur, j0
                if minv[j] < delta: delta, j1 = minv[j], j
            for j in range(n+1):
                if used[j]: u[p[j]] += delta; v[j] -= delta
                else: minv[j] -= delta
            j0 = j1
            if p[j0] == 0: break
        while j0: j1 = way[j0]; p[j0] = p[j1]; j0 = j1
    a = [-1]*n
    for j in range(1, n+1):
        if p[j]: a[p[j]-1] = j-1
    return a

def um_dist(a, b):
    dz, dy, dx = (a[0]-b[0])*VOXEL[0], (a[1]-b[1])*VOXEL[1], (a[2]-b[2])*VOXEL[2]
    return (dz*dz+dy*dy+dx*dx) ** 0.5

def link_tracklets(frames):
    """frames: list per-t of [(z,y,x), ...]. Returns (nodes, edges). Causal, adjacent-frame only."""
    nodes, edges, nid = [], [], [0]
    def new(t, z, y, x):
        nid[0] += 1; nodes.append({"id": nid[0], "t": t, "z": z, "y": y, "x": x}); return nid[0]
    prev = [(new(0, *c)) for c in frames[0]]
    lut = {i: c for i, c in zip(prev, frames[0])}
    for t in range(1, len(frames)):
        cur = [new(t, *c) for c in frames[t]]
        P, Q = sorted(prev), sorted(cur)
        N = max(len(P), len(Q))
        C = [[0.]*N for _ in range(N)]
        for i in range(N):
            for j in range(N):
                if i < len(P) and j < len(Q):
                    d = um_dist(lut[P[i]], lut[Q[j]])
                    C[i][j] = d if d <= MATCH_UM else 1e9
                elif i < len(P): C[i][j] = MATCH_UM + 1e-9
        for i, j in enumerate(hungarian(C)):
            if i < len(P) and 0 <= j < len(Q) and C[i][j] <= MATCH_UM:
                edges.append([P[i], Q[j]])
        # fork proposals (EXP-0005 r10): unmatched target within PROPOSE_UM of a linked target
        have_in = {v for _, v in edges}
        out_of = {}
        for u, v in edges: out_of.setdefault(u, []).append(v)
        by_id = {n["id"]: n for n in nodes}
        cur_pos = {i: lut.get(i, tuple(frames[t][cur.index(i)])) for i in cur}
        for u, vs in sorted(out_of.items()):
            if len(vs) != 1: continue
            v = vs[0]; vp = (by_id[v]["z"], by_id[v]["y"], by_id[v]["x"])
            best = None
            for w in cur:
                if w == v or w in have_in: continue
                d = um_dist(vp, cur_pos[w])
                if d <= PROPOSE_UM and (best is None or d < best[0]): best = (d, w)
            if best: edges.append([u, best[1]]); have_in.add(best[1])
        prev = cur; lut.update(zip(cur, frames[t]))
    return nodes, sorted(edges)

In [ ]:
def detect(volume_t):
    """TODO: replace with real detector + bundled weights.
    Demo fallback: global threshold + connected components on one timepoint."""
    from scipy.ndimage import label
    bw = volume_t > volume_t.mean() + 3 * volume_t.std()
    lab, n = label(bw)
    out = []
    for i in range(1, n + 1):
        z, y, x = [int(c) for c in np.argwhere(lab == i).mean(axis=0)]
        out.append((z, y, x))
    return out

def demo_frames():
    rng = np.random.default_rng(0)
    base = [(32, 100, 100), (32, 150, 120), (40, 90, 90)]
    drift = [(1, 1, 0), (0, -1, 0), (0, 0, 2)]
    frames = []
    for t in range(5):
        frames.append([(z + dz*t + int(rng.integers(-1, 2)), y + dy*t, x + dx*t)
                       for (z, y, x), (dz, dy, dx) in zip(base, drift)])
    return frames

In [ ]:
import csv, os
rows, rid = [], [0]
def emit_dataset(name, nodes, edges):
    for n in sorted(nodes, key=lambda d: d["id"]):
        rows.append([rid[0], name, "node", n["id"], n["t"], n["z"], n["y"], n["x"], -1, -1]); rid[0] += 1
    for u, v in sorted(map(tuple, edges)):
        rows.append([rid[0], name, "edge", -1, -1, -1, -1, -1, u, v]); rid[0] += 1

t_start = time.time()
if MODE == "demo":
    nodes, edges = link_tracklets(demo_frames())
    emit_dataset("demo", nodes, edges)
    print(f"demo: {len(nodes)} nodes, {len(edges)} edges")
else:
    import zarr
    ds_names = sorted(d for d in os.listdir(TEST_ROOT) if d.endswith(".zarr"))
    per_video = []
    for d in ds_names:
        t0 = time.time()
        vol = zarr.open_group(os.path.join(TEST_ROOT, d), mode="r")["0"]
        frames = [detect(vol[t]) for t in range(vol.shape[0])]
        nodes, edges = link_tracklets(frames)
        emit_dataset(d[:-len(".zarr")], nodes, edges)
        per_video.append(time.time() - t0)
        print(f"{d}: {len(nodes)} nodes {len(edges)} edges {per_video[-1]:.1f}s", flush=True)
    import numpy as _np
    print(f"videos={len(ds_names)} mean_s={_np.mean(per_video):.1f} max_s={max(per_video):.1f} total_h={(time.time()-T0)/3600:.2f}")

with open(OUT_CSV, "w", newline="") as f:
    w = csv.writer(f); w.writerow(["id","dataset","row_type","node_id","t","z","y","x","source_id","target_id"]); w.writerows(rows)
print("wrote", OUT_CSV, len(rows), "rows")